# OpenAI Responses API Function Calling: What Changes Across 3 Levels

This notebook shows the difference between:

1. **Tool schema only**
2. **Tool schema + Python function implementation**
3. **Full round-trip function calling flow**

The goal is to make it very clear what each layer does, and what it does **not** do.

---

## Big idea

A tool schema tells the model:

- the function name
- what the function does
- what arguments it may generate

But the schema alone does **not** execute anything.

Your Python function implementation gives your application something real to run.

The full round-trip flow is what connects the model's tool request to your code and then sends the result back to the model.


## Install dependencies

Run this once if needed:

```bash
pip install --upgrade openai python-dotenv
```


In [1]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Add it to your environment or .env file.")

print("API key loaded successfully.")


API key loaded successfully.


In [2]:
from openai import OpenAI

client = OpenAI()
print("OpenAI client initialized.")


OpenAI client initialized.


# Part 1 Tool schema only

This section defines a tool schema and sends it to the model.

### What this gives you
- The model knows a tool named `get_weather` exists
- The model may return a **function call request**

### What this does **not** give you
- No local Python function is executed
- No final weather result is produced automatically
- Your app still has to do the work


In [4]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, such as Dallas or Boston"
                },
                "unit": {
                    "type": "string",
                    "enum": ["fahrenheit", "celsius"],
                    "description": "Temperature unit"
                }
            },
            "required": ["location", "unit"],
            "additionalProperties": False
        },
        "strict": True
    }
]

tools


[{'type': 'function',
  'name': 'get_weather',
  'description': 'Get the current weather for a city.',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string',
     'description': 'City name, such as Dallas or Boston'},
    'unit': {'type': 'string',
     'enum': ['fahrenheit', 'celsius'],
     'description': 'Temperature unit'}},
   'required': ['location', 'unit'],
   'additionalProperties': False},
  'strict': True}]

In [5]:
response_schema_only = client.responses.create(
    model="gpt-5",
    input="What is the weather in Dallas in fahrenheit?",
    tools=tools
)

response_schema_only


Response(id='resp_097e6c36f594b9e40069e151cf51bc8195a66f7e40bb9c0f7c', created_at=1776374223.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_097e6c36f594b9e40069e151d00a5c819586eb27c747e1e9b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"location":"Dallas","unit":"fahrenheit"}', call_id='call_NqorJSr0EmfpvvCvQ0JBoalD', name='get_weather', type='function_call', id='fc_097e6c36f594b9e40069e151d16c688195af10cbc95532157f', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City name, such as Dallas or Boston'}, 'unit': {'type': 'string', 'enum': ['fahrenheit', 'celsius'], 'description': 'Temperature unit'}}, 'required': ['location',

In [6]:
for item in response_schema_only.output:
    print("TYPE:", item.type)
    print(item)
    print("-" * 80)


TYPE: reasoning
ResponseReasoningItem(id='rs_097e6c36f594b9e40069e151d00a5c819586eb27c747e1e9b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None)
--------------------------------------------------------------------------------
TYPE: function_call
ResponseFunctionToolCall(arguments='{"location":"Dallas","unit":"fahrenheit"}', call_id='call_NqorJSr0EmfpvvCvQ0JBoalD', name='get_weather', type='function_call', id='fc_097e6c36f594b9e40069e151d16c688195af10cbc95532157f', namespace=None, status='completed')
--------------------------------------------------------------------------------


### Observation

At this point, the model may return a `function_call` item with arguments.

That means:

- the model is **asking your application** to run the tool
- the model is **not** running your Python code by itself

So if you stop here, you usually have only a **tool request**, not the final answer grounded in tool output.


# Part 2 Tool schema + Python function implementation

Now we add the actual Python function.

This gives your application a real function it can run.

### What this adds
- A local function implementation exists
- Your app can now execute something meaningful

### What this still does **not** do by itself
- The model still does not automatically call your Python function
- You still need orchestration code to:
  - detect the function call
  - parse arguments
  - run the function
  - send the output back


In [7]:
def get_weather(location: str, unit: str = "fahrenheit"):
    """Mock local function implementation for demo purposes."""
    fake_weather_db = {
        "dallas": {"temperature_f": 78, "temperature_c": 26, "condition": "sunny"},
        "irving": {"temperature_f": 77, "temperature_c": 25, "condition": "partly cloudy"},
        "mckinney": {"temperature_f": 76, "temperature_c": 24, "condition": "warm"},
        "boston": {"temperature_f": 61, "temperature_c": 16, "condition": "cloudy"},
    }

    record = fake_weather_db.get(location.strip().lower())
    if record is None:
        return {
            "found": False,
            "location": location,
            "message": f"No mock weather data available for {location}."
        }

    temp = record["temperature_f"] if unit == "fahrenheit" else record["temperature_c"]

    return {
        "found": True,
        "location": location,
        "unit": unit,
        "temperature": temp,
        "condition": record["condition"]
    }

# This is just a normal Python function call done by your app
print(get_weather("Dallas", "fahrenheit"))


{'found': True, 'location': 'Dallas', 'unit': 'fahrenheit', 'temperature': 78, 'condition': 'sunny'}


In [8]:
response_with_schema_and_code = client.responses.create(
    model="gpt-5",
    input="What is the weather in Boston in celsius?",
    tools=tools
)

for item in response_with_schema_and_code.output:
    print("TYPE:", item.type)
    print(item)
    print("-" * 80)


TYPE: reasoning
ResponseReasoningItem(id='rs_0dfa1e59f4ff8c590069e15289a7e88195b5640f1ee2d7864b', summary=[], type='reasoning', content=None, encrypted_content=None, status=None)
--------------------------------------------------------------------------------
TYPE: function_call
ResponseFunctionToolCall(arguments='{"location":"Boston","unit":"celsius"}', call_id='call_p2CLNNNpay7F4kgtS5K1Lt9P', name='get_weather', type='function_call', id='fc_0dfa1e59f4ff8c590069e1528b0e9c8195897efe41ac7ce311', namespace=None, status='completed')
--------------------------------------------------------------------------------


### Observation

Even though `get_weather()` now exists in Python, the model response still just returns a tool request.

Why?

Because defining the function in Python is not enough. Your application must still connect the model's request to your Python function.

That connection happens in the next part.


# Part 3 Full round-trip function calling flow

This is the complete pattern.

### Step A
Send the user question and tool schema to the model.

### Step B
Read any `function_call` items returned by the model.

### Step C
Run the matching Python function locally in your app.

### Step D
Send the tool result back using `function_call_output`.

### Step E
Get the final natural-language answer from the model.


In [9]:
first_response = client.responses.create(
    model="gpt-5",
    input="What is the weather in Dallas in fahrenheit?",
    tools=tools
)

first_response


Response(id='resp_06abf2e2dbf8126d0069e1529bbcb88194bcb6b15f458e34c5', created_at=1776374427.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_06abf2e2dbf8126d0069e1529c6948819493966692a4a21fda', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"location":"Dallas","unit":"fahrenheit"}', call_id='call_IDo6SrirlQar80fk2Yi8gu8I', name='get_weather', type='function_call', id='fc_06abf2e2dbf8126d0069e1529d1cb081948b75600585c31599', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City name, such as Dallas or Boston'}, 'unit': {'type': 'string', 'enum': ['fahrenheit', 'celsius'], 'description': 'Temperature unit'}}, 'required': ['location',

In [11]:
tool_outputs = []

for item in first_response.output:
    if item.type == "function_call" and item.name == "get_weather":
        arguments = json.loads(item.arguments)
        result = get_weather(**arguments)

        print("Arguments from model:", arguments)
        print("Local function result:", result)

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result)
        })

tool_outputs


Arguments from model: {'location': 'Dallas', 'unit': 'fahrenheit'}
Local function result: {'found': True, 'location': 'Dallas', 'unit': 'fahrenheit', 'temperature': 78, 'condition': 'sunny'}


[{'type': 'function_call_output',
  'call_id': 'call_IDo6SrirlQar80fk2Yi8gu8I',
  'output': '{"found": true, "location": "Dallas", "unit": "fahrenheit", "temperature": 78, "condition": "sunny"}'}]

In [12]:
if tool_outputs:
    second_response = client.responses.create(
        model="gpt-5",
        previous_response_id=first_response.id,
        input=tool_outputs,
        tools=tools
    )

    print(second_response.output_text)
else:
    print("No function call was returned.")


Currently in Dallas: 78°F and sunny.


## Why the third version is the real working pattern

The third version is the one that turns:
- model intent
- your local business logic
- tool output
- final model response

into one complete workflow.

Without this round-trip:
- the model can request a tool
- but your app will never actually use the tool result


# Side-by-side summary


In [13]:
comparison = {
    "tool_schema_only": {
        "defines_tool": True,
        "has_python_function": False,
        "model_can_request_tool": True,
        "tool_executes": False,
        "final_tool_grounded_answer": False
    },
    "tool_schema_plus_python_function": {
        "defines_tool": True,
        "has_python_function": True,
        "model_can_request_tool": True,
        "tool_executes": False,
        "final_tool_grounded_answer": False
    },
    "full_round_trip": {
        "defines_tool": True,
        "has_python_function": True,
        "model_can_request_tool": True,
        "tool_executes": True,
        "final_tool_grounded_answer": True
    }
}

comparison


{'tool_schema_only': {'defines_tool': True,
  'has_python_function': False,
  'model_can_request_tool': True,
  'tool_executes': False,
  'final_tool_grounded_answer': False},
 'tool_schema_plus_python_function': {'defines_tool': True,
  'has_python_function': True,
  'model_can_request_tool': True,
  'tool_executes': False,
  'final_tool_grounded_answer': False},
 'full_round_trip': {'defines_tool': True,
  'has_python_function': True,
  'model_can_request_tool': True,
  'tool_executes': True,
  'final_tool_grounded_answer': True}}

# Reusable helper

This wraps the full pattern into one function.


In [14]:
def ask_with_function_calling(question: str):
    first = client.responses.create(
        model="gpt-5",
        input=question,
        tools=tools
    )

    function_outputs = []

    for item in first.output:
        if item.type == "function_call" and item.name == "get_weather":
            args = json.loads(item.arguments)
            result = get_weather(**args)
            function_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result)
            })

    if function_outputs:
        second = client.responses.create(
            model="gpt-5",
            previous_response_id=first.id,
            input=function_outputs,
            tools=tools
        )
        return second.output_text

    return first.output_text


In [15]:
print(ask_with_function_calling("Tell me the weather in McKinney in celsius."))


In McKinney, it’s currently 24°C and warm.


# Final takeaway

- **Tool schema only**: the model knows about the tool, but nothing gets executed.
- **Tool schema + Python function**: your code exists, but the model still does not run it automatically.
- **Full round-trip**: your app reads the tool request, runs the function, sends back the result, and gets the final answer.

This is the key conceptual difference to remember when using custom function calling with the Responses API.
